# Use time-travel  利用时间旅行
LangGraph 支持通过检查点进行时间旅行：
- `Replay`重放 ：从之前的检查点重新尝试。
- `Fork`分支 ：从先前的检查点创建分支，并修改其状态以探索替代路径。

两者都通过从之前的检查点恢复执行来实现。检查点之前的节点不会重新执行（结果已经保存）。检查点之后的节点会重新执行，包括所有 LLM 调用、API 请求和中断 （这可能会导致不同的结果）。

## Replay  重播
重放操作会重新执行节点，而不仅仅是从缓存读取数据。LLM 调用、API 请求和中断都会再次触发，并可能返回不同的结果。从最后一个检查点（没有 next 节点）重放则不会执行任何操作。

使用 `get_state_history` 找到要重放的检查点，然后使用该检查点的配置调用 invoke ：

In [ ]:
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict, NotRequired
import uuid

class State(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]


def generate_topic(state: State):
    return {"topic": "socks in the dryer"}


def write_joke(state: State):
    return {"joke": f"Why do {state['topic']} disappear? They elope!"}


checkpointer = InMemorySaver()
graph = (
    StateGraph(State)
    .add_node("generate_topic", generate_topic)
    .add_node("write_joke", write_joke)
    .add_edge(START, "generate_topic")
    .add_edge("generate_topic", "write_joke")
    .compile(checkpointer=checkpointer)
)

# Step 1: Run the graph
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = graph.invoke({}, config)

# Step 2: Find a checkpoint to replay from
history = list(graph.get_state_history(config))
# History is in reverse chronological order
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

# Step 3: Replay from a specific checkpoint
# Find the checkpoint before write_joke
before_joke = next(s for s in history if s.next == ("write_joke",))
replay_result = graph.invoke(None, before_joke.config)
# write_joke re-executes (runs again), generate_topic does not
replay_result

## Fork  叉
Fork 函数会从过去的检查点创建一个状态已修改的新分支。调用 `update_state` 更新之前的检查点即可创建 fork 分支，然后 invoke 并传入 None 即可继续执行。

`update_state` 不会回滚线程。它会创建一个新的检查点，该检查点从指定的点分支而出。原始的执行历史记录保持不变。

In [ ]:
# Find checkpoint before write_joke
history = list(graph.get_state_history(config))
before_joke = next(s for s in history if s.next == ("write_joke",))

# Fork: update state to change the topic
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
)

# Resume from the fork — write_joke re-executes with the new topic
fork_result = graph.invoke(None, fork_config)
print(fork_result["joke"])  # A joke about chickens, not socks

### From a specific node  从特定节点
调用 `update_state` 时，值会通过指定节点的写入器（包括 `reducer` ）应用。检查点会将该节点记录为已执行更新，执行会从该节点的后续节点恢复。

默认情况下，LangGraph 会根据检查点的版本历史记录推断 as_node 。当从特定检查点 fork 时，这种推断几乎总是正确的。

显式指定 as_node 情况：
- 并行分支 ：多个节点在同一步骤中更新了状态，LangGraph 无法确定哪个是最后更新的（ InvalidUpdateError ）。
- 没有执行历史记录 ：在新线程上设置状态（常见于测试中）。
- 跳过节点 ：将 as_node 设置为后面的节点，使图认为该节点已经运行过。

In [ ]:
# graph: generate_topic -> write_joke

# Treat this update as if generate_topic produced it.
# Execution resumes at write_joke (the successor of generate_topic).
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
    as_node="generate_topic",
)
fork_result = graph.invoke(None, fork_config)
fork_result

## Interrupts  打断
如果你的图使用 interrupt 来实现人机交互工作流程，那么在时间旅行期间中断总是会被重新触发。包含中断的节点会重新执行，并且 interrupt() 会暂停以等待新的 Command(resume=...) 。

In [ ]:
from langgraph.types import interrupt, Command

class State(TypedDict):
    value: list[str]

def ask_human(state: State):
    answer = interrupt("What is your name?")
    return {"value": [f"Hello, {answer}!"]}

def final_step(state: State):
    return {"value": ["Done"]}

graph = (
    StateGraph(State)
    .add_node("ask_human", ask_human)
    .add_node("final_step", final_step)
    .add_edge(START, "ask_human")
    .add_edge("ask_human", "final_step")
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# First run: hits interrupt
graph.invoke({"value": []}, config)
# Resume with answer
graph.invoke(Command(resume="Alice"), config)

# Replay from before ask_human
history = list(graph.get_state_history(config))
before_ask = [s for s in history if s.next == ("ask_human",)][-1]

replay_result = graph.invoke(None, before_ask.config)
# Pauses at interrupt — waiting for new Command(resume=...)

# Fork from before ask_human
fork_config = graph.update_state(before_ask.config, {"value": ["forked"]})
fork_result = graph.invoke(None, fork_config)
# Pauses at interrupt — waiting for new Command(resume=...)

# Resume the forked interrupt with a different answer
graph.invoke(Command(resume="Bob"), fork_config)
# Result: {"value": ["forked", "Hello, Bob!", "Done"]}

## Subgraphs  子图
利用子图进行时间旅行取决于子图是否拥有自己的检查点。这决定了你可以进行时间旅行的检查点的粒度。

将子图的 checkpointer=True 即可为其创建独立的检查点历史记录。这会在子图的每个步骤创建检查点，允许您从子图内部的特定点进行时间旅行——例如，在两个中断之间。

使用 `get_state` 并 `subgraphs=True` ，可以访问子图自身的检查点配置，然后从该配置创建子图：

In [ ]:
# Subgraph with its own checkpointer
subgraph = (
    StateGraph(State)
    .add_node("step_a", step_a)       # Has interrupt()
    .add_node("step_b", step_b)       # Has interrupt()
    .add_edge(START, "step_a")
    .add_edge("step_a", "step_b")
    .compile(checkpointer=True)  # Own checkpoint history
)

graph = (
    StateGraph(State)
    .add_node("subgraph_node", subgraph)
    .add_edge(START, "subgraph_node")
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# Run until step_a interrupt
graph.invoke({"value": []}, config)

# Resume step_a -> hits step_b interrupt
graph.invoke(Command(resume="Alice"), config)

# Get the subgraph's own checkpoint (between step_a and step_b)
parent_state = graph.get_state(config, subgraphs=True)
sub_config = parent_state.tasks[0].state.config

# Fork from the subgraph checkpoint
fork_config = graph.update_state(sub_config, {"value": ["forked"]})
result = graph.invoke(None, fork_config)
# step_b re-executes, step_a's result is preserved